# Day 08 — CPU Scheduling, Visualized

One CPU, many ready processes. The OS must pick who runs next. Different **policies**
make different trade-offs — let's render them as **Gantt charts** on one workload.

## The workload

| Process | Arrival | Burst |
|---|--:|--:|
| P1 | 0 | 5 |
| P2 | 1 | 3 |
| P3 | 2 | 1 |

A **process** is a running program with its own memory; the **burst** is how much CPU
time it needs before it's done. (Threads share a process's memory — more on Day 11.)

In [ ]:
from collections import deque

WORK = [('P1', 0, 5), ('P2', 1, 3), ('P3', 2, 1)]  # (pid, arrival, burst)

def fcfs(work):
    segs, t = [], 0
    for pid, arr, burst in sorted(work, key=lambda p: (p[1], p[0])):
        s = max(t, arr); segs.append((pid, s, s + burst)); t = s + burst
    return segs

def sjf(work):
    rem, t, segs = list(work), 0, []
    while rem:
        ready = [p for p in rem if p[1] <= t]
        if not ready:
            t = min(p[1] for p in rem); continue
        pid, arr, burst = min(ready, key=lambda p: (p[2], p[1], p[0]))
        rem.remove((pid, arr, burst))
        s = max(t, arr); segs.append((pid, s, s + burst)); t = s + burst
    return segs

def round_robin(work, q):
    arr = sorted(work, key=lambda p: (p[1], p[0]))
    rem = {p[0]: p[2] for p in work}; ready, segs, t, i = deque(), [], 0, 0
    def board(upto):
        nonlocal i
        while i < len(arr) and arr[i][1] <= upto:
            ready.append(arr[i][0]); i += 1
    board(0)
    while ready or i < len(arr):
        if not ready:
            t = arr[i][1]; board(t); continue
        pid = ready.popleft(); run = min(q, rem[pid])
        segs.append((pid, t, t + run)); t += run; rem[pid] -= run
        board(t)
        if rem[pid] > 0: ready.append(pid)
    return segs

for name, segs in [('FCFS', fcfs(WORK)), ('SJF', sjf(WORK)),
                   ('RR q=2', round_robin(WORK, 2))]:
    print(f'{name:8}', segs)

## Gantt charts: who holds the CPU, when

In [ ]:
import matplotlib.pyplot as plt

policies = {'FCFS': fcfs(WORK), 'SJF': sjf(WORK), 'Round-Robin (q=2)': round_robin(WORK, 2)}
colors = {'P1': '#4C72B0', 'P2': '#DD8452', 'P3': '#55A868'}

fig, axes = plt.subplots(len(policies), 1, figsize=(8, 5), sharex=True)
for ax, (name, segs) in zip(axes, policies.items()):
    for pid, s, e in segs:
        ax.broken_barh([(s, e - s)], (0, 1), facecolors=colors[pid], edgecolor='white')
        ax.text((s + e) / 2, 0.5, pid, va='center', ha='center', color='white', fontsize=9)
    ax.set_title(name, loc='left', fontsize=10); ax.set_yticks([])
axes[-1].set_xlabel('time (ticks)')
plt.tight_layout(); plt.show()

## How they compare on average waiting time

**Waiting = turnaround − burst** (time spent ready but not running).

In [ ]:
def avg_wait(work, segs):
    completion = {}
    for pid, s, e in segs:
        completion[pid] = max(completion.get(pid, 0), e)
    waits = [completion[pid] - arr - burst for pid, arr, burst in work]
    return sum(waits) / len(waits)

names = list(policies)
waits = [avg_wait(WORK, policies[n]) for n in names]
plt.figure(figsize=(6, 3.5))
plt.bar(names, waits, color=['#4C72B0', '#55A868', '#DD8452'])
plt.ylabel('avg waiting time'); plt.title('Lower is snappier')
for x, w in enumerate(waits): plt.text(x, w, f'{w:.2f}', ha='center', va='bottom')
plt.show()

## Takeaways

- **FCFS**: simple, but a long job makes short jobs wait (convoy effect).
- **SJF**: best average wait, but needs to know burst lengths and can starve long jobs.
- **Round-Robin**: fair & responsive; the quantum tunes responsiveness vs. overhead.

**Now go implement all three** in `homework.py` (with proper metrics) and run `pytest -q`.